# FORGE — pipeline di produzione: MySQL → forge() → validazione hold-out → persistenza regole

Questo notebook esegue l'intero ciclo per scoprire regole di trading con **forgedge/FORGE** in modo
disciplinato dal punto di vista statistico:

1. **Ingestione** delle candele da MySQL (parametrizzata per `ticker` e `timeframe`).
2. **Censura** della serie storica: l'ultima porzione (periodo di hold-out) viene esclusa da ogni fase
   di discovery/selezione dei parametri — non la vede nessuno fino al punto 4.
3. **Esecuzione di `forge()`** sul solo periodo di training, eventualmente con più preset
   (`sniper` / `balanced` / `sweep` / `burst`), così da coprire profili di ricerca diversi.
4. **Validazione out-of-sample** di ogni regola sopravvissuta (verdetto `EDGE`/`PARTIAL-EDGE`) nel
   periodo di hold-out, usando il **motore di backtest integrato di forge** (`forgedge.rule_discovery.run_backtest`)
   sugli stessi identici parametri operativi scelti in fase di training — nessuna riselezione dei
   parametri sul hold-out, per evitare qualunque leakage. La validazione è **per periodo calendariale**
   (mese o trimestre), non un unico backtest aggregato: la regola deve attivarsi in una quota minima
   dei periodi e, tra quelli in cui si attiva, chiudere in profitto in una quota minima di essi.
5. **Persistenza** di ogni regola che supera anche il hold-out: un file **JSON** con i parametri
   operativi (direzione, `sell_pct`, `target_h`, ...) e le metriche di performance (win rate, profit
   factor, expectancy, ...) sia in-sample/walk-forward che di hold-out.

Il notebook è testato sulla fixture `tests/fixtures/ADA_1D_TRAIN.parquet` (candele giornaliere ADA) —
impostare `DATA_SOURCE = "mysql"` per usarlo in produzione con un vero database.

> **Invariante da rispettare**: il periodo di hold-out non deve mai influenzare le soglie degli eventi
> (M1), il target derivato (M2) o la selezione dei parametri operativi (M3). Viene usato *soltanto* per
> un singolo backtest finale, a parametri congelati — esattamente il pattern "monitorare un edge
> pubblicato su dati nuovi" descritto nella skill `forgedge` (pattern 5), applicato qui al proprio
> hold-out invece che a candele realmente future.


## 0. Setup

In [1]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from forgedge import forge, forge_preset, config_report, summary_report
from forgedge.rule_discovery import run_backtest
from forgedge.deployment import (
    KpiRecipe, minimal_kpi_recipe, verify_kpi_recipe, export_rules, PromotionGateConfig,
)

pd.set_option("display.width", 140)


## 1. Configurazione

Tutti i parametri della run sono qui. `DATA_SOURCE` seleziona la sorgente dati:

- `"mysql"` — candele grezze lette da MySQL, parametrizzate per `TICKER`/`TIMEFRAME`; viene poi
  costruita la KPI Table con `forgedge.build_features`/`candle_features`.
- `"fixture"` — usa direttamente la fixture di test del repo (`tests/fixtures/ADA_1D_TRAIN.parquet`),
  già una KPI Table pronta. Usata per testare il notebook end-to-end senza un database.


In [2]:
# --- sorgente dati ----------------------------------------------------------
DATA_SOURCE = "fixture"          # "mysql" | "fixture"
TICKER      = "ADAUSDC"
TIMEFRAME   = "1D"               # "1H" o "1D" — i due timeframe supportati da questo notebook

# --- connessione MySQL (usata solo se DATA_SOURCE == "mysql") ---------------
MYSQL_CONFIG = dict(
    host="localhost",
    port=3306,
    user="forge",
    password="",          # preferire una variabile d'ambiente in produzione
    database="market_data",
)
# Nome tabella per timeframe — adattare allo schema del proprio DB.
MYSQL_TABLE_BY_TIMEFRAME = {
    "1H": "candles_1h",
    "1D": "candles_1d",
}

# --- fixture di test (usata solo se DATA_SOURCE == "fixture") ---------------
FIXTURE_PATH = "../tests/fixtures/ADA_1D_TRAIN.parquet"

# --- KPI opt-in --------------------------------------------------------
# build_features() disabilita di default ATR, MACD, CCI, Williams %R,
# Stochastic %K/%D, WMA, TRIMA, ADX, Aroon e Williams A/D (richiedono
# high/low oltre a close, o sono semplicemente extra rispetto al set base).
# ENABLE_OPTIONAL_KPI=True li attiva tutti, più il pattern categorico
# candle_pattern di pattern_features() (anch'esso opt-in).
ENABLE_OPTIONAL_KPI = True
OPTIONAL_KPI_INDICATORS = [
    "atr", "macd", "cci", "willr", "stochastic", "wma", "trima", "adx", "aroon", "ad",
]

# --- hold-out -----------------------------------------------------------
HOLDOUT_FRACTION = 0.20   # ultima frazione della storia censurata dalla discovery

# --- preset da provare in sequenza (vedi skill forgedge: 'sniper' e 'sweep'
# richiedono storie lunghe su 1D e possono essere scartati dal coherence check) ---
PRESETS = ["burst", "balanced", "sniper", "sweep"]

# --- soglie di sopravvivenza sul hold-out: per PERIODO, non sull'aggregato ---
# Il hold-out viene diviso in periodi calendariali (mesi o trimestri) e la
# regola è valutata periodo per periodo, non su un unico backtest aggregato:
# un singolo draw direzionale (es. un bear market di 6 mesi) pesa come un solo
# "voto" nel criterio di attivazione/sopravvivenza, non può più da solo far
# superare o fallire la regola. Parametrico in tutti e tre gli assi:
HOLDOUT_PERIOD_FREQ          = "M"    # "M" (mese) o "Q" (trimestre)
MIN_PERIOD_ACTIVATION_RATE   = 0.50   # la regola deve attivarsi (>=1 trade) in almeno questa quota dei periodi di hold-out
MIN_PERIOD_SURVIVAL_RATE     = 0.75   # tra i periodi in cui si attiva, deve chiudere in profitto in almeno questa quota
# Con pochi periodi disponibili (es. HOLDOUT_FRACTION piccola -> pochi mesi di
# hold-out) le due quote collassano naturalmente su valori estremi (0%, 50%,
# 100%, ...) e il criterio degenera in "ha funzionato nell'unico/pochi periodi
# osservati" — lo stesso giudizio binario della validazione aggregata di
# prima, senza bisogno di un ramo di codice separato per i due casi.

# --- output -------------------------------------------------------------
OUTPUT_DIR = Path("./discovered_rules")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Ingestione candele da MySQL

Query parametrizzata su `ticker`/`timeframe`. Richiede `sqlalchemy` + `pymysql`
(`pip install sqlalchemy pymysql`). Lo schema atteso per la tabella candele è
`(open_time, open, high, low, close, volume, symbol)` — adattare la query se il proprio schema
differisce. `open_time` può essere epoch (ms/s) o già un datetime: la conversione è gestita sotto.


In [3]:
def load_candles_from_mysql(ticker: str, timeframe: str, config: dict = MYSQL_CONFIG) -> pd.DataFrame:
    """Carica candele OHLCV grezze da MySQL per un ticker/timeframe, ordinate cronologicamente."""
    from sqlalchemy import create_engine, text

    table = MYSQL_TABLE_BY_TIMEFRAME.get(timeframe)
    if table is None:
        raise ValueError(f"Nessuna tabella configurata per timeframe={timeframe!r} "
                          f"(vedi MYSQL_TABLE_BY_TIMEFRAME)")

    conn_str = (
        f"mysql+pymysql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['database']}"
    )
    engine = create_engine(conn_str)
    query = text(f"""
        SELECT open_time, open, high, low, close, volume
        FROM {table}
        WHERE symbol = :ticker
        ORDER BY open_time ASC
    """)
    with engine.connect() as conn:
        raw = pd.read_sql(query, conn, params={"ticker": ticker})

    if raw.empty:
        raise RuntimeError(f"Nessuna candela trovata per {ticker} / {timeframe} in {table}")

    # open_time può essere epoch ms, epoch s o già datetime: normalizza.
    if np.issubdtype(raw["open_time"].dtype, np.number):
        unit = "ms" if raw["open_time"].iloc[0] > 10**12 else "s"
        raw["open_dt"] = pd.to_datetime(raw["open_time"], unit=unit)
    else:
        raw["open_dt"] = pd.to_datetime(raw["open_time"])

    raw = raw.drop(columns=["open_time"]).sort_values("open_dt").reset_index(drop=True)
    return raw


def load_fixture(path: str) -> pd.DataFrame:
    """Carica la fixture di test (già una KPI Table completa, non candele grezze)."""
    df = pd.read_parquet(path)
    return df.sort_values("open_dt").reset_index(drop=True)


if DATA_SOURCE == "mysql":
    raw_candles = load_candles_from_mysql(TICKER, TIMEFRAME)
    print(f"MySQL: {len(raw_candles)} candele {TICKER}/{TIMEFRAME} "
          f"({raw_candles['open_dt'].min()} -> {raw_candles['open_dt'].max()})")
elif DATA_SOURCE == "fixture":
    raw_candles = load_fixture(FIXTURE_PATH)
    print(f"Fixture: {len(raw_candles)} righe ({raw_candles['open_dt'].min()} -> {raw_candles['open_dt'].max()})")
else:
    raise ValueError(f"DATA_SOURCE sconosciuta: {DATA_SOURCE!r}")


Fixture: 882 righe (2024-01-01 00:00:00 -> 2026-05-31 00:00:00)


## 3. Costruzione della KPI Table (con indicatori opt-in)

Se la sorgente è MySQL (candele grezze OHLCV) costruiamo l'intero set di indicatori con
`build_features`/`candle_features` di forgedge, usando una config che — se `ENABLE_OPTIONAL_KPI` —
attiva anche gli opt-in. Se la sorgente è la fixture di test, è già una KPI Table con gli indicatori
*di base* pronti: per evitare di ricalcolarli (e scontrarsi con le colonne già presenti), le passiamo
comunque attraverso `build_features()` ma con una config che attiva **solo** gli indicatori opt-in,
così da *aggiungere* le nuove colonne senza toccare quelle esistenti. In entrambi i casi aggiungiamo
anche `pattern_features()` (il pattern candlestick categorico, anch'esso opt-in).

`summary_report()` valida la qualità dei dati prima di procedere (non blocca mai da sola — decidiamo
noi cosa fare con `has_critical`).


In [4]:
import copy
from forgedge import build_features, candle_features, pattern_features
from forgedge.kpi_builder import DEFAULT_CONFIG

full_config = copy.deepcopy(DEFAULT_CONFIG)       # indicatori base + opt-in (per candele grezze)
optin_only_config = copy.deepcopy(DEFAULT_CONFIG)  # solo opt-in (per una KPI Table già costruita)
for name in optin_only_config:
    optin_only_config[name]["enabled"] = name in OPTIONAL_KPI_INDICATORS

if ENABLE_OPTIONAL_KPI:
    for name in OPTIONAL_KPI_INDICATORS:
        full_config[name]["enabled"] = True

if DATA_SOURCE == "mysql":
    kpi = build_features(raw_candles, config=full_config, timestamp_col="open_dt")
    kpi = candle_features(kpi)
else:
    kpi = raw_candles  # già una KPI Table con gli indicatori di base
    if ENABLE_OPTIONAL_KPI:
        kpi = build_features(kpi, config=optin_only_config, timestamp_col="open_dt")

if ENABLE_OPTIONAL_KPI:
    kpi = pattern_features(kpi)

rep = summary_report(kpi, timeframe=TIMEFRAME, return_report=True, verbose=False)
print(rep.one_line() if hasattr(rep, "one_line") else rep)
if rep.has_critical:
    raise ValueError("La KPI Table ha problemi critici di qualità — correggere prima di procedere.")

print(f"\nKPI table: {len(kpi)} bar, {kpi['open_dt'].min()} -> {kpi['open_dt'].max()}, "
      f"{kpi.shape[1]} colonne (KPI opt-in: {'attivi' if ENABLE_OPTIONAL_KPI else 'disattivi'})")


[WARN] OHLC: range intrabar >100% (max/min>2) in 1 barre; [WARN] 1 rendimenti anomali (MAD z>8) — possibili print errati o salti di scala; [WARN] 1 barre con |rendimento| > 50% — estremo; reale su asset molto volatili (es. crypto), ma da verificare se non atteso

KPI table: 882 bar, 2024-01-01 00:00:00 -> 2026-05-31 00:00:00, 51 colonne (KPI opt-in: attivi)


## 4. Censura del periodo di hold-out

Si taglia la serie cronologicamente: l'ultima `HOLDOUT_FRACTION` dei bar non viene mai passata a
`forge()` — né come dato di training né come dato di selezione (M1/M2/M3 operano solo su `train_df`).
Verrà riunita a `train_df` **soltanto** al passo 6, per un singolo backtest finale a parametri
congelati (mai per ri-allenare soglie o target).


In [5]:
n = len(kpi)
split_idx = int(round(n * (1 - HOLDOUT_FRACTION)))

train_df = kpi.iloc[:split_idx].reset_index(drop=True)
holdout_df = kpi.iloc[split_idx:].reset_index(drop=True)
holdout_start = holdout_df["open_dt"].iloc[0]

print(f"train   : {len(train_df):4d} bar  ({train_df['open_dt'].iloc[0]} -> {train_df['open_dt'].iloc[-1]})")
print(f"hold-out: {len(holdout_df):4d} bar  ({holdout_start} -> {holdout_df['open_dt'].iloc[-1]})")


train   :  706 bar  (2024-01-01 00:00:00 -> 2025-12-06 00:00:00)
hold-out:  176 bar  (2025-12-07 00:00:00 -> 2026-05-31 00:00:00)


## 5. Esecuzione di `forge()` sul solo training, per ogni preset

Per ciascun preset: si costruisce la tripla di config con `forge_preset()`, si verifica la coerenza
con `config_report()` **prima** di lanciare la pipeline (pattern 6 della skill `forgedge`) e si salta
il preset se il check trova un `FAIL` — su una storia 1D corta (come questa fixture dopo aver tolto
l'hold-out) è normale che `sniper`/`sweep` vengano scartati (vedi pitfall #8 della skill): non è un
bug, è il controllo di coerenza che rifiuta correttamente una sessione che non potrebbe sostenere il
test richiesto.


In [6]:
preset_results = {}

for preset in PRESETS:
    disc_cfg, alpha_cfg, rd_cfg = forge_preset(preset, timeframe=TIMEFRAME, asset=TICKER)
    rep = config_report(disc_cfg, alpha_cfg, rd_cfg, kpi=train_df, timeframe=TIMEFRAME)

    if rep.has_critical:
        print(f"[{preset:10s}] SKIPPED — coherence FAIL: {rep.one_line()}")
        continue

    print(f"[{preset:10s}] coherence OK — running forge()...")
    try:
        result = forge(
            train_df,
            ticker=TICKER,
            timeframe=TIMEFRAME,
            event_discovery_config=disc_cfg,
            alpha_config=alpha_cfg,
            rule_discovery_config=rd_cfg,
            progress=False,
        )
    except ValueError as exc:
        print(f"[{preset:10s}] forge() raised: {exc}")
        continue

    n_edges = sum(1 for _, r in result.rule_responses if r.is_edge)
    print(f"[{preset:10s}] done — {len(result.rule_responses)} regole valutate, {n_edges} tradeable")
    preset_results[preset] = result

if not preset_results:
    raise RuntimeError("Nessun preset ha prodotto una run coerente su questa finestra di training.")


[burst     ] coherence OK — running forge()...


[burst     ] done — 441 regole valutate, 7 tradeable
[balanced  ] coherence OK — running forge()...


[balanced  ] done — 1226 regole valutate, 118 tradeable
[sniper    ] SKIPPED — coherence FAIL: [WARN] m1_is_window_too_short: la finestra di discovery IS (23.5 mesi = 23.5 mesi di storia x train_ratio=1) non raggiunge i gate_params.min_episodes=10 al tasso configurato min_tpm=0.3: servono 53.3 mesi con margine di Poisson al 95 % (4.44 anni). Un candidato esattamente al floor di frequenza non avrà mai abbastanza episodi per superare il Criterion 2 su questa storia — non un errore, ma la ragione per cui una IS lunga rende meno di quanto sembri: abbassare min_episodes a <= 4, oppure alzare min_tpm, oppure allungare la storia.; [FAIL] oos_span_too_short: walk_forward.min_train_months=31 copre l'intero span disponibile (24 mesi): non resta nessuna finestra di test. La finestra è derivata da criteria.min_tpm=0.528 — servono 10 trade con margine di Poisson 95 % (#173). Alzare min_tpm a >= 1.36 perché selezione e test stiano entrambi nello span, oppure allungare la storia.
[sweep     ] SKIPPED

## 6. Raccolta, deduplica per overlap e benchmark naive

Si raccolgono tutti i responsi con verdetto `EDGE`/`PARTIAL-EDGE` da ogni preset eseguito. Due
accorgimenti, nati da un difetto osservato nel primo giro di test su questa stessa fixture, sono
**necessari** prima di guardare il hold-out, altrimenti la validazione è fuorviante:

1. **Deduplica per overlap, non per testo.** La composizione AND guidata dal grado (M2) produce
   spesso varianti con `event_expression` testualmente diverse ma che condividono la stessa
   componente dominante e si attivano quasi sulle stesse barre (pitfall #20 della skill `forgedge`).
   Confrontare solo la stringa dell'espressione le conta come scoperte indipendenti quando non lo
   sono. Si raggruppano quindi i candidati per **somiglianza di Jaccard** tra le rispettive serie di
   attivazione calcolate **sul solo training** (mai sul hold-out, per non introdurre leakage), e si
   tiene un solo rappresentante per cluster (il migliore per profit factor walk-forward OOS di
   training).
2. **Benchmark naive direzionale (riferimento informativo).** Su una finestra di hold-out con un
   drift di mercato marcato in una direzione, qualunque regola orientata in quella direzione può
   apparire profittevole solo grazie al beta di mercato, non a un pattern specifico. Si calcola quindi
   anche una baseline "sempre attiva" che usa **esattamente gli stessi parametri operativi**
   (direzione, `target_h`, `sell_pct`, ...) ma nessuna condizione sull'evento, e la si riporta accanto
   a ogni regola — non più come gate assoluto (quel ruolo lo svolge ora la validazione per periodo di
   §7, che è per costruzione più robusta a un singolo draw direzionale: un bear/bull market lungo
   quanto l'intero hold-out pesa come un solo periodo su molti, non come l'intero verdetto).


In [7]:
# mappa event_id -> EventCandidate, serve più avanti per ricostruire il segnale sul hold-out
candidates_by_id = {}
for result in preset_results.values():
    for cand in result.candidates:
        candidates_by_id[cand.event_id] = cand

pool = {}  # (direzione, expression) -> (preset, contract, response, oos_pf)
for preset, result in preset_results.items():
    for contract, response in result.rule_responses:
        if not response.is_edge:
            continue
        key = (contract.derived_target.direction, contract.event_expression)
        oos_pf = (
            response.walk_forward.oos_summary.profit_factor
            if response.walk_forward else float("nan")
        )
        prev = pool.get(key)
        if prev is None or (not np.isnan(oos_pf) and oos_pf > prev[3]):
            pool[key] = (preset, contract, response, oos_pf)

print(f"{len(pool)} regole tradeable distinte (per testo) su {len(preset_results)} preset (pre hold-out)")


123 regole tradeable distinte (per testo) su 2 preset (pre hold-out)


In [8]:
# --- deduplica per overlap di attivazione, calcolata SOLO sul training -----
JACCARD_DUP_THRESHOLD = 0.6

pool_items = list(pool.items())
train_activations = [
    candidates_by_id[contract.event_candidate_id].apply(train_df).fillna(0).astype(bool).to_numpy()
    for _, (preset, contract, response, _) in pool_items
]

parent = list(range(len(pool_items)))
def _find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def _union(i, j):
    ri, rj = _find(i), _find(j)
    if ri != rj:
        parent[ri] = rj

for i in range(len(train_activations)):
    for j in range(i + 1, len(train_activations)):
        a, b = train_activations[i], train_activations[j]
        union_n = np.logical_or(a, b).sum()
        jaccard = np.logical_and(a, b).sum() / union_n if union_n else 0.0
        if jaccard >= JACCARD_DUP_THRESHOLD:
            _union(i, j)

clusters = {}
for i, item in enumerate(pool_items):
    clusters.setdefault(_find(i), []).append(item)

pool = {}
for members in clusters.values():
    best_key, best_val = max(members, key=lambda kv: kv[1][3] if not np.isnan(kv[1][3]) else -np.inf)
    pool[best_key] = best_val

print(f"-> {len(pool)} regole distinte dopo la deduplica per overlap "
      f"(Jaccard>={JACCARD_DUP_THRESHOLD} sull'attivazione di training)")


-> 21 regole distinte dopo la deduplica per overlap (Jaccard>=0.6 sull'attivazione di training)


## 7. Validazione hold-out per periodo, con il backtest integrato di forge

Per ogni regola del pool (dedupicato): si ricostruisce il segnale booleano sull'intera serie
(train + hold-out) con `EventCandidate.apply()` — la stessa identica logica di soglie fissata in
fase di discovery — e si esegue **`forgedge.rule_discovery.run_backtest`** (il motore di backtest di
Modulo 3) ristretto alle sole barre di apertura del periodo di hold-out
(`timerange_from=holdout_start`), usando esattamente i parametri operativi già scelti in training
(`response.validated_rule.params`, `return_trades=True` per avere il ledger per-trade). In parallelo
si esegue, con gli stessi parametri, la baseline naive "sempre attiva" descritta sopra — tenuta come
riferimento informativo, non come gate.

Nessuna riottimizzazione sul hold-out: è un singolo backtest a parametri congelati, esattamente il
pattern di monitoraggio raccomandato dalla skill `forgedge` per un edge già pubblicato.

**Il gate di sopravvivenza è per periodo calendariale** (`HOLDOUT_PERIOD_FREQ`), non sull'aggregato:

1. Si enumerano tutti i periodi (mesi o trimestri) coperti dalla finestra di hold-out.
2. Ogni trade del ledger viene assegnato al suo periodo in base alla data di `fill_dt` (apertura
   effettiva della posizione). Un periodo è **attivo** se contiene almeno un trade.
3. Un periodo attivo **sopravvive** se la somma dei `net_pct_gain` dei suoi trade è positiva.
4. `activation_rate = periodi attivi / periodi totali`; `survival_rate = periodi sopravvissuti /
   periodi attivi`. La regola è promossa se `activation_rate >= MIN_PERIOD_ACTIVATION_RATE` **e**
   `survival_rate >= MIN_PERIOD_SURVIVAL_RATE`.

Con pochi periodi nel hold-out (`HOLDOUT_FRACTION` piccola) le due quote assumono naturalmente valori
pochi e grossolani (es. 1/3, 2/3, 3/3) invece di collassare in un giudizio binario artificiale: la
stessa formula regge sia un hold-out lungo (molti periodi, stima granulare) sia uno corto (pochi
periodi, stima grossolana ma coerente) senza un ramo di codice a parte.


In [9]:
full_df = pd.concat([train_df, holdout_df], ignore_index=True).sort_values("open_dt").reset_index(drop=True)
SIGNAL_COL = "__holdout_signal__"
NAIVE_COL = "__naive_signal__"
naive_cache = {}  # (direction, buy_drop_pct, sell_pct, target_h, buy_type) -> BacktestSummary

all_holdout_periods = pd.period_range(
    start=holdout_df["open_dt"].min(), end=holdout_df["open_dt"].max(), freq=HOLDOUT_PERIOD_FREQ,
)
N_HOLDOUT_PERIODS = len(all_holdout_periods)
print(f"hold-out: {N_HOLDOUT_PERIODS} periodo/i di frequenza '{HOLDOUT_PERIOD_FREQ}' "
      f"({all_holdout_periods[0]} -> {all_holdout_periods[-1]})")


def _naive_baseline(params, df):
    """Baseline: sempre attiva, stessi parametri operativi, nessuna condizione sull'evento."""
    key = (params.direction, params.buy_drop_pct, params.sell_pct, params.target_h, params.buy_type)
    if key not in naive_cache:
        work = df.copy()
        work[NAIVE_COL] = True
        naive_cache[key], _ = run_backtest(
            work, NAIVE_COL, params,
            timerange_from=str(holdout_start), timerange_to=None,
            timestamp_col="open_dt", return_trades=True,
        )
    return naive_cache[key]


def _period_breakdown(trades: pd.DataFrame) -> pd.DataFrame:
    """Net gain per periodo di hold-out, a partire dal ledger di run_backtest."""
    if trades.empty:
        return pd.DataFrame(columns=["period", "net_gain", "n_trades"])
    per_trade = trades.copy()
    per_trade["period"] = pd.to_datetime(per_trade["fill_dt"]).dt.to_period(HOLDOUT_PERIOD_FREQ)
    grouped = per_trade.groupby("period")["net_pct_gain"].agg(["sum", "count"])
    return grouped.rename(columns={"sum": "net_gain", "count": "n_trades"}).reset_index()


holdout_results = []  # lista di dict con tutto il necessario per il passo 8

for (direction, expr), (preset, contract, response, train_oos_pf) in pool.items():
    candidate = candidates_by_id.get(contract.event_candidate_id)
    if candidate is None:
        print(f"SKIP {contract.alpha_id}: EventCandidate non trovato nel pool")
        continue

    params = response.validated_rule.params

    work = full_df.copy()
    work[SIGNAL_COL] = candidate.apply(work).fillna(0).astype(bool)

    holdout_summary, holdout_trades = run_backtest(
        work, SIGNAL_COL, params,
        timerange_from=str(holdout_start), timerange_to=None,
        timestamp_col="open_dt", return_trades=True,
    )
    naive_summary = _naive_baseline(params, full_df)  # riferimento informativo, non gate

    period_stats = _period_breakdown(holdout_trades)
    n_active = len(period_stats)
    n_survived = int((period_stats["net_gain"] > 0).sum()) if n_active else 0
    activation_rate = n_active / N_HOLDOUT_PERIODS if N_HOLDOUT_PERIODS else float("nan")
    survival_rate = n_survived / n_active if n_active else float("nan")

    survives = (
        activation_rate >= MIN_PERIOD_ACTIVATION_RATE
        and not np.isnan(survival_rate)
        and survival_rate >= MIN_PERIOD_SURVIVAL_RATE
    )

    print(f"{contract.alpha_id:30s} [{preset:9s}] train={response.verdict:13s} -> "
          f"periodi attivi={n_active:2d}/{N_HOLDOUT_PERIODS} (att.={activation_rate:4.0%}) "
          f"sopravvissuti={n_survived:2d}/{n_active if n_active else 0} (surv.={survival_rate:4.0%})  "
          f"[info: trades={holdout_summary.total_trades:3d} pf={holdout_summary.profit_factor:6.2f} "
          f"exp={holdout_summary.expectancy:+.4f} vs naive {direction} exp={naive_summary.expectancy:+.4f}]  "
          f"-> {'SURVIVES' if survives else 'rejected'}")

    holdout_results.append(dict(
        preset=preset, contract=contract, response=response, candidate=candidate,
        holdout_summary=holdout_summary, holdout_trades=holdout_trades,
        naive_summary=naive_summary, period_stats=period_stats,
        n_active_periods=n_active, n_survived_periods=n_survived,
        activation_rate=activation_rate, survival_rate=survival_rate,
        survives=survives,
    ))

survivors = [r for r in holdout_results if r["survives"]]
print(f"\n{len(survivors)}/{len(holdout_results)} regole sopravvivono alla validazione hold-out per periodo "
      f"(attivazione>={MIN_PERIOD_ACTIVATION_RATE:.0%}, sopravvivenza>={MIN_PERIOD_SURVIVAL_RATE:.0%})")


hold-out: 6 periodo/i di frequenza 'M' (2025-12 -> 2026-05)
ALPHA-ADAUSDC-1D-261001-209    [burst    ] train=PARTIAL-EDGE  -> periodi attivi= 6/6 (att.=100%) sopravvissuti= 2/6 (surv.= 33%)  [info: trades= 14 pf=  0.79 exp=-0.0102 vs naive long exp=-0.0399]  -> rejected
ALPHA-ADAUSDC-1D-261001-263    [burst    ] train=PARTIAL-EDGE  -> periodi attivi= 6/6 (att.=100%) sopravvissuti= 4/6 (surv.= 67%)  [info: trades= 14 pf=  0.95 exp=-0.0012 vs naive long exp=-0.0423]  -> rejected
ALPHA-ADAUSDC-1D-261001-3011   [balanced ] train=PARTIAL-EDGE  -> periodi attivi= 3/6 (att.= 50%) sopravvissuti= 0/3 (surv.=  0%)  [info: trades=  7 pf=  0.41 exp=-0.0244 vs naive long exp=-0.0278]  -> rejected
ALPHA-ADAUSDC-1D-261001-1755   [burst    ] train=PARTIAL-EDGE  -> periodi attivi= 6/6 (att.=100%) sopravvissuti= 1/6 (surv.= 17%)  [info: trades= 13 pf=  0.23 exp=-0.0204 vs naive long exp=-0.0073]  -> rejected
ALPHA-ADAUSDC-1D-261001-070    [balanced ] train=PARTIAL-EDGE  -> periodi attivi= 3/6 (att.= 50%

ALPHA-ADAUSDC-1D-261001-3982   [balanced ] train=PARTIAL-EDGE  -> periodi attivi= 3/6 (att.= 50%) sopravvissuti= 0/3 (surv.=  0%)  [info: trades=  5 pf=  0.00 exp=-0.0385 vs naive long exp=-0.0214]  -> rejected


ALPHA-ADAUSDC-1D-261001-3990   [balanced ] train=PARTIAL-EDGE  -> periodi attivi= 3/6 (att.= 50%) sopravvissuti= 0/3 (surv.=  0%)  [info: trades=  9 pf=  0.00 exp=-0.0608 vs naive long exp=-0.0145]  -> rejected
ALPHA-ADAUSDC-1D-261001-4213   [balanced ] train=PARTIAL-EDGE  -> periodi attivi= 0/6 (att.=  0%) sopravvissuti= 0/0 (surv.=nan%)  [info: trades=  0 pf=  0.00 exp=+nan vs naive short exp=+0.0205]  -> rejected

2/21 regole sopravvivono alla validazione hold-out per periodo (attivazione>=50%, sopravvivenza>=75%)


## 8. Persistenza delle regole sopravvissute

Per ogni regola sopravvissuta scriviamo un JSON con:

- **parametri operativi**: espressione dell'evento, direzione, modalità di entrata, `buy_drop_pct`,
  `buy_delay_bar`, `sell_pct`, `target_h`, fee (da `ValidatedRule.to_dict()`);
- **parametri di performance**: metriche complete (`total_trades`, `profit_factor`, `win_rate_pct`,
  `expectancy`, `tpm_mu`, concorrenza, ...) per il training (in-sample e walk-forward OOS), per il
  backtest di hold-out aggregato e per la baseline naive di hold-out — quest'ultima resta nel JSON
  così chi legge la regola può verificare da sé quanto margine c'è sopra il semplice beta direzionale;
- **validazione per periodo**: `activation_rate`/`survival_rate` effettivi, il conteggio di periodi
  attivi/sopravvissuti su quelli totali, e il dettaglio `net_gain`/`n_trades` di ogni periodo — il
  criterio di sopravvivenza effettivamente applicato, non solo l'aggregato;
- **ricetta KPI minimale** (`minimal_kpi_recipe`, issue [#296](https://github.com/mattcond/forgedge/issues/296)):
  la configurazione minima di `build_features()`/`candle_features()`/`pattern_features()`/
  `lag_features()` necessaria per ricalcolare **solo** le colonne KPI che l'evento referenzia
  (indicatore, colonna, finestra), ora prodotta da **`forgedge.deployment.minimal_kpi_recipe()`** —
  non più da codice locale al notebook. La libreria legge
  `EventCandidate.components[*].source_cols` per costruzione (mai per induzione sulla naming
  convention), gestisce ricorsivamente gli AND-composti e gli eventi `CustomEvent`, e risolve ogni
  combinazione indicatore/colonna/periodo con un'unica mappa inversa invece del probing
  brute-force del prototipo originale.

Subito dopo aver calcolato la ricetta, la **verifichiamo** con
**`forgedge.deployment.verify_kpi_recipe()`**: ricostruisce una KPI Table ridotta dalle sole candele
OHLC, vi riapplica `EventCandidate.apply()` e confronta l'attivazione risultante con quella sulla KPI
Table completa — qui, esplicitamente, **sulla sola finestra di hold-out** (`evaluation_mask`), dato
che la libreria stessa documenta di non applicare alcuna euristica di warm-up di default.

Viene scritto anche un indice CSV riassuntivo con una riga per regola sopravvissuta.


In [10]:
def _clean(obj):
    """Rende l'oggetto serializzabile in JSON: NaN -> null, numpy -> python nativi."""
    if isinstance(obj, dict):
        return {k: _clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_clean(v) for v in obj]
    if isinstance(obj, float) and np.isnan(obj):
        return None
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    return obj


index_rows = []

holdout_mask = full_df["open_dt"] >= holdout_start

for r in survivors:
    contract, response, candidate = r["contract"], r["response"], r["candidate"]

    minimal_recipe = minimal_kpi_recipe(candidate)  # forgedge.deployment, #296
    verification = verify_kpi_recipe(
        candidate, minimal_recipe, full_df,
        evaluation_mask=holdout_mask, timestamp_col="open_dt",
    )
    recipe_verified = verification.matches

    record = {
        "rule_id": contract.alpha_id,
        "ticker": TICKER,
        "timeframe": TIMEFRAME,
        "preset": r["preset"],
        "verdict_train": response.verdict,
        "event_expression": contract.event_expression,
        "operational_params": response.validated_rule.to_dict(),
        "performance": {
            "train_in_sample": response.in_sample_summary.to_dict(),
            "train_walk_forward_oos": (
                response.walk_forward.oos_summary.to_dict() if response.walk_forward else None
            ),
            "holdout": r["holdout_summary"].to_dict(),
            "holdout_naive_baseline": r["naive_summary"].to_dict(),
        },
        "holdout_period_validation": {
            "period_freq": HOLDOUT_PERIOD_FREQ,
            "n_total_periods": N_HOLDOUT_PERIODS,
            "n_active_periods": r["n_active_periods"],
            "n_survived_periods": r["n_survived_periods"],
            "activation_rate": r["activation_rate"],
            "survival_rate": r["survival_rate"],
            "min_activation_rate": MIN_PERIOD_ACTIVATION_RATE,
            "min_survival_rate": MIN_PERIOD_SURVIVAL_RATE,
            "periods": [
                {"period": str(row.period), "net_gain": row.net_gain, "n_trades": int(row.n_trades)}
                for row in r["period_stats"].itertuples()
            ],
        },
        "holdout_window": {
            "from": str(holdout_start),
            "to": str(holdout_df["open_dt"].iloc[-1]),
        },
        "minimal_kpi_recipe": minimal_recipe.to_dict(),
        "minimal_kpi_recipe_verification": verification.to_dict(),
    }
    record = _clean(record)

    out_path = OUTPUT_DIR / f"{contract.alpha_id}.json"
    out_path.write_text(json.dumps(record, indent=2, default=str))
    print(f"scritto {out_path}  [ricetta KPI minimale verificata: "
          f"{'OK — eventi identici' if recipe_verified else 'FALLITA — vedi unresolved_columns'}]")

    index_rows.append({
        "rule_id": contract.alpha_id,
        "preset": r["preset"],
        "direction": response.validated_rule.params.direction,
        "verdict_train": response.verdict,
        "target_h": response.validated_rule.params.target_h,
        "sell_pct": response.validated_rule.params.sell_pct,
        "n_active_periods": r["n_active_periods"],
        "n_total_periods": N_HOLDOUT_PERIODS,
        "activation_rate": r["activation_rate"],
        "survival_rate": r["survival_rate"],
        "holdout_trades": r["holdout_summary"].total_trades,
        "holdout_profit_factor": r["holdout_summary"].profit_factor,
        "holdout_expectancy": r["holdout_summary"].expectancy,
        "naive_holdout_expectancy": r["naive_summary"].expectancy,
        "minimal_kpi_recipe_verified": recipe_verified,
        "json_path": str(out_path),
    })

if index_rows:
    index_df = pd.DataFrame(index_rows).sort_values("survival_rate", ascending=False)
    index_df.to_csv(OUTPUT_DIR / "survivors_index.csv", index=False)
    display(index_df)
else:
    print("Nessuna regola ha superato la validazione hold-out per periodo con le soglie correnti "
          "(MIN_PERIOD_ACTIVATION_RATE / MIN_PERIOD_SURVIVAL_RATE) — "
          "è un esito legittimo quanto trovarne: significa che nessun pattern scoperto in training "
          "ha retto con sufficiente regolarità sul periodo futuro censurato.")


scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-1130.json  [ricetta KPI minimale verificata: OK — eventi identici]
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-3564.json  [ricetta KPI minimale verificata: OK — eventi identici]


,rule_id,preset,direction,verdict_train,target_h,sell_pct,n_active_periods,n_total_periods,activation_rate,survival_rate,holdout_trades,holdout_profit_factor,holdout_expectancy,naive_holdout_expectancy,minimal_kpi_recipe_verified,json_path
0,ALPHA-ADAUSDC-1D-261001-1130,balanced,short,PARTIAL-EDGE,48,0.1985,4,6,0.666667,1.0,21,9999.0,0.124216,0.111695,True,discovered_rules/ALPHA-ADAUSDC-1D-261001-1130....
1,ALPHA-ADAUSDC-1D-261001-3564,balanced,short,PARTIAL-EDGE,12,0.2264,3,6,0.500000,1.0,4,9999.0,0.035195,0.025638,True,discovered_rules/ALPHA-ADAUSDC-1D-261001-3564....


## 8b. Test funzionale: `forgedge.deployment` end-to-end (issue #296)

Approfittiamo di avere almeno una regola sopravvissuta per testare la nuova funzionalità anche nel
suo percorso "ufficiale" di produzione, non solo nella chiamata diretta usata sopra:

1. **Round-trip di serializzazione** — `KpiRecipe.to_json()` seguito da `KpiRecipe.from_json()` deve
   restituire un oggetto identico a quello di partenza (lo stesso contratto che la skill `forgedge`
   documenta: `KpiRecipe.from_json(recipe.to_json()) == recipe`).
2. **`export_rules()`** — il percorso di persistenza "ufficiale" del modulo deployment, che scrive
   `.pkl` + `.yaml` + `.kpi_recipe.json` per ogni contratto esportato e verifica la ricetta da sé,
   usando `ForgeResult.event_frame` (cioè il **solo periodo di training**, non l'hold-out — un ambito
   di verifica diverso e complementare rispetto a quello di §8, che verifica esplicitamente
   sull'hold-out). Usiamo `promotable_only=False`: il gate di `export_rules()`
   (`PromotionGateConfig`, pensato per duplicati/isolamento/rotation-null) è un criterio diverso dal
   nostro gate per periodo, quindi qui vogliamo solo esportare le regole che **noi** abbiamo già
   selezionato, non ri-filtrarle con un criterio diverso.
3. **Cross-check** — il file `.kpi_recipe.json` scritto da `export_rules()` deve contenere la stessa
   `build_features_config` calcolata a mano in §8 per la stessa regola, a conferma che i due percorsi
   (chiamata diretta vs. `export_rules()`) sono coerenti.


In [11]:
if not survivors:
    print("Nessun sopravvissuto su cui testare export_rules() in questa run "
          "(vedi §7) — il test funzionale viene saltato, non indica un problema della libreria.")
else:
    best = survivors[0]
    best_recipe = minimal_kpi_recipe(best["candidate"])

    # 1. round-trip di serializzazione
    roundtrip_recipe = KpiRecipe.from_json(best_recipe.to_json())
    print(f"[1] KpiRecipe round-trip (to_json -> from_json): "
          f"{'OK — identica' if roundtrip_recipe == best_recipe else 'FALLITA — vedi sotto'}")
    if roundtrip_recipe != best_recipe:
        print("    originale:", best_recipe.to_dict())
        print("    round-trip:", roundtrip_recipe.to_dict())

    # 2. export_rules() — percorso ufficiale del modulo deployment
    EXPORT_DIR = OUTPUT_DIR / "export_rules_test"
    export_result = preset_results[best["preset"]]
    manifest = export_rules(
        [export_result], EXPORT_DIR,
        promotable_only=False,   # il nostro gate per periodo ha già selezionato le regole
        include_kpi_recipe=True,
    )
    survivor_ids = {r["contract"].alpha_id for r in survivors}
    manifest_survivors = manifest[manifest["alpha_id"].isin(survivor_ids)]
    print(f"\n[2] export_rules() ha scritto {len(manifest)} contratti tradeable in {EXPORT_DIR} "
          f"({len(manifest_survivors)} dei quali sono nostri sopravvissuti):")
    display(manifest_survivors[["alpha_id", "verdict", "pkl_path", "yaml_path",
                                 "kpi_recipe_path", "kpi_recipe_verified"]])
    print("    (kpi_recipe_verified qui è sul solo TRAINING — result.event_frame —"
          " non sull'hold-out come in §8: è normale un valore diverso da quello di §8 per la "
          "stessa regola, sono due verifiche su finestre diverse.)")

    # 3. cross-check tra il file scritto da export_rules() e il calcolo diretto di §8
    best_row = manifest_survivors[manifest_survivors["alpha_id"] == best["contract"].alpha_id].iloc[0]
    exported_payload = json.loads(Path(best_row["kpi_recipe_path"]).read_text())
    exported_recipe = KpiRecipe.from_dict(exported_payload)
    print(f"\n[3] Cross-check ricetta diretta vs. file di export_rules() per {best['contract'].alpha_id}: "
          f"{'OK — identiche' if exported_recipe == best_recipe else 'DIVERSE — vedi sotto'}")
    if exported_recipe != best_recipe:
        print("    diretta:   ", best_recipe.to_dict())
        print("    esportata: ", exported_recipe.to_dict())


[1] KpiRecipe round-trip (to_json -> from_json): OK — identica



[2] export_rules() ha scritto 118 contratti tradeable in discovered_rules/export_rules_test (2 dei quali sono nostri sopravvissuti):


,alpha_id,verdict,pkl_path,yaml_path,kpi_recipe_path,kpi_recipe_verified
9,ALPHA-ADAUSDC-1D-261001-1130,PARTIAL-EDGE,discovered_rules/export_rules_test/ALPHA-ADAUS...,discovered_rules/export_rules_test/ALPHA-ADAUS...,discovered_rules/export_rules_test/ALPHA-ADAUS...,False
74,ALPHA-ADAUSDC-1D-261001-3564,PARTIAL-EDGE,discovered_rules/export_rules_test/ALPHA-ADAUS...,discovered_rules/export_rules_test/ALPHA-ADAUS...,discovered_rules/export_rules_test/ALPHA-ADAUS...,False


    (kpi_recipe_verified qui è sul solo TRAINING — result.event_frame — non sull'hold-out come in §8: è normale un valore diverso da quello di §8 per la stessa regola, sono due verifiche su finestre diverse.)

[3] Cross-check ricetta diretta vs. file di export_rules() per ALPHA-ADAUSDC-1D-261001-1130: OK — identiche


## 9. Riepilogo

- Regole tradeable scoperte in training (pre hold-out), dedupicate per overlap: vedi §6.
- Regole che hanno superato anche il hold-out **per periodo** (parametri congelati, nessuna
  riselezione, attivazione/sopravvivenza su mesi o trimestri distinti, non sull'aggregato): vedi §7-8.
- Ogni regola sopravvissuta è persistita in `discovered_rules/<rule_id>.json` con parametri operativi,
  performance complete (training + hold-out + baseline naive informativa), il dettaglio
  `activation_rate`/`survival_rate` periodo-per-periodo e la ricetta KPI minimale di
  `forgedge.deployment` (§8, issue #296); `discovered_rules/survivors_index.csv` è l'indice
  riassuntivo.
- §8b verifica la stessa funzionalità anche sul percorso ufficiale `export_rules()` del modulo
  deployment, con un cross-check tra le due strade.

**La validazione per periodo riduce ma non azzera il rischio di un singolo hold-out**: con pochi
periodi disponibili le due quote restano grossolane (es. 2/3 invece di una stima su 20+ osservazioni),
e un pattern che sopravvive solo appena sopra `MIN_PERIOD_SURVIVAL_RATE` va trattato come indizio, non
come prova. Prima di considerare una regola operativa, ripetere la validazione su più finestre di
hold-out non sovrapposte e su più asset con lo stesso pattern.

Per usare questo notebook in produzione su un vero database:

1. Impostare `DATA_SOURCE = "mysql"` e compilare `MYSQL_CONFIG` / `MYSQL_TABLE_BY_TIMEFRAME`.
2. Cambiare `TICKER` / `TIMEFRAME` (`"1H"` o `"1D"`) in §1.
3. Rivedere `PRESETS` e le soglie di hold-out in base alla lunghezza di storia realmente disponibile.
